# Tutorial on How to Use FIM-PP

This notebook shows the recommended Hugging Face workflow for the point-process model from {cite}`fim_pp`: load the pretrained model with `AutoModel.from_pretrained(...)`, download a small example dataset from the Hub, prepare the context/inference tensors, visualize the inferred intensities, and finish with a fine-tuning command template.


In [1]:
%matplotlib inline

import warnings
from pathlib import Path

import torch
from huggingface_hub import snapshot_download
from point_process_tutorial_helper import load_hawkes_tensors, move_to_device, plot_intensity_comparison, prepare_hawkes_batch
from transformers import AutoModel


warnings.filterwarnings("ignore")
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    if torch.backends.mps.is_available():
        print("MPS is not yet supported for this FIM-PP tutorial path; using CPU instead.")
    device = torch.device("cpu")
device

MPS is not yet supported for this FIM-PP tutorial path; using CPU instead.


device(type='cpu')

## Load the Pretrained Model

The standardized user-facing path is now the Transformers AutoModel interface.


In [ ]:
model_root = Path(snapshot_download(repo_id="FIM4Science/FIM-PP", repo_type="model"))
model = AutoModel.from_pretrained(model_root, trust_remote_code=True)
model = model.to(device)
model.eval()
model_root

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

model-checkpoint.pth:   0%|          | 0.00/64.6M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/64.5M [00:00<?, ?B/s]

__init__.py:   0%|          | 0.00/134 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

model_architecture.txt: 0.00B [00:00, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/64.6M [00:00<?, ?B/s]

train_parameters.yaml: 0.00B [00:00, ?B/s]

modeling_hawkes.py:   0%|          | 0.00/66.0 [00:00<?, ?B/s]

## Download Example Data

The tutorial dataset is stored as raw tensors on Hugging Face. We download the snapshot and load the `.pt` files directly.


In [ ]:
dataset_root = Path(snapshot_download(repo_id="FIM4Science/10D-Hawkes", repo_type="dataset"))
dataset_root

In [ ]:
tensors = load_hawkes_tensors(dataset_root)
sorted(tensors)

## Build a Context / Inference Batch

We hold out a single path for inference and use the remaining paths as context. The helper also builds a dense evaluation grid for plotting the intensity curves between events.


In [ ]:
batch = prepare_hawkes_batch(tensors, sample_idx=0, inference_path_idx=0, num_points_between_events=10)
batch = move_to_device(batch, device)

for key, value in batch.items():
    if torch.is_tensor(value):
        print(f"{key}: {tuple(value.shape)}")
    else:
        print(f"{key}: {value}")

## Run Zero-Shot Inference


In [ ]:
with torch.no_grad():
    output = model(batch)

sorted(output.keys())

In [ ]:
fig = plot_intensity_comparison(output, batch, path_idx=0)
fig

## Fine-Tuning Starting from FIM-PP

A short fine-tuning run can be started with the existing Hawkes entrypoint. The point-process checkpoint is the initialization source, while the dataset can be either a local tensor folder or an EasyTPP dataset id.

For this tutorial, the `10D-Hawkes` snapshot is used for inference and visualization. The fine-tuning CLI was smoke-tested with `easytpp/retweet`, which matches the expected training layout directly.

Use the downloaded model directory from the earlier `snapshot_download(...)` call as `--resume_model`. The script accepts either that directory or a specific file inside it such as `model-checkpoint.pth`.

```bash
python scripts/hawkes/fim_finetune.py \
  --config configs/train/hawkes/david.yaml \
  --dataset easytpp/retweet \
  --resume_model /absolute/path/to/downloaded/FIM-PP \
  --save_dir results/finetuned_cdiff \
  --epochs 200 \
  --val-every 10
```

If you use the notebook variable directly, `--resume_model` should point to `model_root`.

The fine-tuned model is written under `save_dir/<dataset_name>/<timestamp>/`. With the command above, a run on `easytpp/retweet` will be stored in a directory like `results/finetuned_cdiff/retweet/260401-1430/`, and the exported checkpoint will appear in `best-model/` inside that folder.

If `--save_dir` is omitted, the script defaults to `results/finetuned_cdiff/<dataset_name>/<timestamp>/`.

For local debugging, the lower-level fallback `fim.models.hawkes.FIMHawkes.load_model(...)` is still available, but the primary public workflow should use `AutoModel.from_pretrained(...)`.
